# CS5487 Digits Project In Colab

Use this notebook as the primary execution surface when local CPU and memory are limited. It can now pull the project source from GitHub at startup, while keeping `artifacts/` persistent in Google Drive so batch outputs survive Colab runtime resets.

In [1]:
USE_GOOGLE_DRIVE = True
SYNC_PROJECT_FROM_GITHUB = True
GITHUB_REPO_URL = "https://github.com/MJWade96/CS5487-Project"
GITHUB_REF = None
PROJECT_ROOT = "/content/CS5487-Project"
ARTIFACTS_ROOT = (
    "/content/drive/MyDrive/CS5487 Course Project Code"
    if USE_GOOGLE_DRIVE
    else None
)
GRID_SEARCH_JOBS = 2
# Use one of: None, "light", "heavy", "rbf_only", "random_forest_only", "mlp_only".
BATCH_PRESET = None
# Set False when you only want to inspect or combine finished batch runs.
RUN_EXPERIMENTS = False
SELECTED_TRIAL_NAMES = None
SELECTED_MODEL_NAMES = None
RUN_NAME = None
# Example: ["batch_light", "batch_heavy"]
COMBINE_RUN_NAMES = ["batch_light", "batch_heavy"]

In [2]:
from pathlib import Path
import importlib
import shutil
import subprocess

if USE_GOOGLE_DRIVE:
    colab_drive = importlib.import_module("google.colab.drive")
    colab_drive.mount("/content/drive")

project_root = Path(PROJECT_ROOT).expanduser()
artifacts_root = None if ARTIFACTS_ROOT is None else Path(ARTIFACTS_ROOT).expanduser()

if SYNC_PROJECT_FROM_GITHUB:
    project_root.parent.mkdir(parents=True, exist_ok=True)
    git_dir = project_root / ".git"
    if git_dir.exists():
        subprocess.check_call(["git", "-C", str(project_root), "fetch", "--all", "--tags"])
        subprocess.check_call(["git", "-C", str(project_root), "pull", "--ff-only"])
    elif project_root.exists():
        raise FileExistsError(
            "PROJECT_ROOT already exists but is not a git checkout. "
            "Choose a clean folder, or remove the old manual copy first."
        )
    else:
        subprocess.check_call(["git", "clone", GITHUB_REPO_URL, str(project_root)])

    if GITHUB_REF is not None:
        subprocess.check_call(["git", "-C", str(project_root), "checkout", GITHUB_REF])
elif not project_root.exists():
    raise FileNotFoundError(
        "Project root not found. Enable SYNC_PROJECT_FROM_GITHUB or update PROJECT_ROOT."
    )

required_children = [
    project_root / "requirements.txt",
    project_root / "src",
    project_root / "digits4000_txt",
    project_root / "challenge",
]
missing_children = [str(path.relative_to(project_root)) for path in required_children if not path.exists()]
if missing_children:
    raise FileNotFoundError(
        "Project root is missing required paths: " + ", ".join(missing_children)
    )

if artifacts_root is not None:
    artifacts_root.mkdir(parents=True, exist_ok=True)
    external_artifacts_dir = artifacts_root / "artifacts"
    repo_artifacts_dir = project_root / "artifacts"

    if repo_artifacts_dir.is_symlink():
        if repo_artifacts_dir.resolve() != external_artifacts_dir.resolve():
            repo_artifacts_dir.unlink()
            repo_artifacts_dir.symlink_to(external_artifacts_dir, target_is_directory=True)
    else:
        if repo_artifacts_dir.exists() and not external_artifacts_dir.exists():
            shutil.copytree(repo_artifacts_dir, external_artifacts_dir)
        elif not external_artifacts_dir.exists():
            external_artifacts_dir.mkdir(parents=True, exist_ok=True)

        if repo_artifacts_dir.exists():
            shutil.rmtree(repo_artifacts_dir)
        repo_artifacts_dir.symlink_to(external_artifacts_dir, target_is_directory=True)

git_head = subprocess.check_output(
    ["git", "-C", str(project_root), "rev-parse", "--short", "HEAD"],
    text=True,
).strip()
print("Project root:", project_root)
print("Artifacts root:", artifacts_root if artifacts_root is not None else project_root)
print("Git revision:", git_head)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root: /content/CS5487-Project
Artifacts root: /content/drive/MyDrive/CS5487 Course Project Code
Git revision: cddc599


In [3]:
import importlib
import os
import subprocess
import sys

os.chdir(project_root)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'])
src_dir = project_root / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

# Force Python to reload the freshly synced checkout instead of reusing stale modules.
importlib.invalidate_caches()
for module_name in list(sys.modules):
    if module_name == 'digits_project' or module_name.startswith('digits_project.'):
        del sys.modules[module_name]

print('Working directory:', project_root)
print('Cleared cached digits_project modules after source sync.')

Working directory: /content/CS5487-Project
Cleared cached digits_project modules after source sync.


In [4]:
import inspect
import json
from pathlib import Path

import pandas as pd

import digits_project.config as project_config
import digits_project.experiment as experiment_module

configure_project = project_config.configure_project
run_project_experiments = getattr(experiment_module, "run_project_experiments", None)
combine_experiment_runs = getattr(experiment_module, "combine_experiment_runs", None)

if run_project_experiments is None:
    raise ImportError(
        "digits_project.experiment is missing run_project_experiments. "
        "Refresh the GitHub checkout in Cell 4 and rerun from Cell 4."
    )

configure_project_signature = inspect.signature(configure_project)
run_project_experiments_signature = inspect.signature(run_project_experiments)


def _call_configure_project(root_dir, grid_search_jobs, run_name):
    kwargs = {
        "root_dir": root_dir,
        "grid_search_jobs": grid_search_jobs,
    }
    if "run_name" in configure_project_signature.parameters:
        kwargs["run_name"] = run_name
    elif run_name is not None:
        print(
            "configure_project in this source tree does not accept run_name; "
            "outputs will use the default artifacts directory."
        )
    return configure_project(**kwargs)


def _call_run_project_experiments(
    root_dir,
    grid_search_jobs,
    selected_trial_names,
    selected_model_names,
    run_name,
 ):
    kwargs = {
        "root_dir": root_dir,
        "grid_search_jobs": grid_search_jobs,
        "selected_trial_names": selected_trial_names,
        "selected_model_names": selected_model_names,
    }
    if "run_name" in run_project_experiments_signature.parameters:
        kwargs["run_name"] = run_name
    elif run_name is not None:
        print(
            "run_project_experiments in this source tree does not accept run_name; "
            "batch outputs will use the default artifacts directory."
        )
    return run_project_experiments(**kwargs)


def _build_summary_frames(final_frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    summary_frame = (
        final_frame.groupby("model", as_index=False)
        .agg(
            mnist_accuracy_mean=("mnist_accuracy", "mean"),
            mnist_accuracy_std=("mnist_accuracy", "std"),
            challenge_accuracy_mean=("challenge_accuracy", "mean"),
            challenge_accuracy_std=("challenge_accuracy", "std"),
            mnist_macro_f1_mean=("mnist_macro_f1", "mean"),
            challenge_macro_f1_mean=("challenge_macro_f1", "mean"),
        )
        .sort_values("mnist_accuracy_mean", ascending=False)
        .reset_index(drop=True)
    )

    email_frame = final_frame[["trial", "model", "mnist_accuracy", "challenge_accuracy"]].copy()
    email_frame["challenge_reference_knn_mean"] = project_config.CHALLENGE_REFERENCE_ACCURACY
    return summary_frame, email_frame


def _merge_result_frames(frames: list[pd.DataFrame], key_columns: list[str]) -> pd.DataFrame:
    non_empty_frames = [frame for frame in frames if not frame.empty]
    if not non_empty_frames:
        raise ValueError("No result rows were available to combine.")
    merged = pd.concat(non_empty_frames, ignore_index=True)
    return merged.drop_duplicates(subset=key_columns, keep="last")


def _resolve_batch_run_names(root_dir: Path, run_names) -> list[str]:
    if run_names is not None:
        if isinstance(run_names, str):
            return [run_names]
        return sorted(set(run_names))

    runs_dir = project_config.ProjectPaths(root_dir).runs_dir
    if not runs_dir.exists():
        raise FileNotFoundError(f"No batch run directory found under {runs_dir}")

    available_names = sorted(path.name for path in runs_dir.iterdir() if path.is_dir())
    if not available_names:
        raise ValueError("No batch runs are available to combine.")
    return available_names


def _save_accuracy_comparison_plot(summary_frame: pd.DataFrame, path: Path) -> None:
    import matplotlib.pyplot as plt

    try:
        import seaborn as sns
    except ImportError:
        sns = None

    plt.figure(figsize=(10, 5))
    if summary_frame.empty:
        plt.text(0.5, 0.5, "No summary rows", ha="center", va="center")
        plt.axis("off")
    elif sns is None:
        plt.bar(summary_frame["model"], summary_frame["mnist_accuracy_mean"], color="#2c7fb8")
        plt.xticks(rotation=30, ha="right")
        plt.ylabel("Mean accuracy on official test set")
        plt.xlabel("Model")
    else:
        sns.barplot(data=summary_frame, x="model", y="mnist_accuracy_mean", color="#2c7fb8")
        plt.ylabel("Mean accuracy on official test set")
        plt.xlabel("Model")
        plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.close()


def _save_result_tables(
    cv_frame: pd.DataFrame,
    final_frame: pd.DataFrame,
    summary_frame: pd.DataFrame,
    email_frame: pd.DataFrame,
    paths,
    source_runs=None,
 ) -> None:
    for path in (
        paths.results_dir,
        paths.figures_dir,
        paths.models_dir,
        paths.predictions_dir,
        paths.per_class_dir,
    ):
        path.mkdir(parents=True, exist_ok=True)

    cv_frame.to_csv(paths.results_dir / "cv_leaderboard.csv", index=False)
    final_frame.to_csv(paths.results_dir / "final_selected_models.csv", index=False)
    summary_frame.to_csv(paths.results_dir / "summary_by_model.csv", index=False)
    email_frame.to_csv(paths.results_dir / "email_summary.csv", index=False)
    _save_accuracy_comparison_plot(summary_frame, paths.figures_dir / "mnist_accuracy_by_model.png")

    protocol_payload = {
        "challenge_reference_accuracy": project_config.CHALLENGE_REFERENCE_ACCURACY,
        "public_material_can_include_challenge": False,
    }
    if source_runs is not None:
        protocol_payload["source_runs"] = list(source_runs)
    (paths.results_dir / "challenge_protocol.json").write_text(
        json.dumps(protocol_payload, indent=2, sort_keys=True),
        encoding="utf-8",
    )


if combine_experiment_runs is None:
    print(
        "combine_experiment_runs is missing in this source tree; "
        "the notebook fallback will be used for batch merging."
    )

    def combine_experiment_runs(root_dir=None, run_names=None) -> dict[str, pd.DataFrame]:
        resolved_root = (
            project_config.ROOT_DIR
            if root_dir is None
            else Path(root_dir).expanduser().resolve()
        )
        selected_run_names = _resolve_batch_run_names(resolved_root, run_names)
        combined_paths = project_config.ProjectPaths(resolved_root)

        cv_frames = []
        final_frames = []
        for run_name in selected_run_names:
            batch_paths = project_config.ProjectPaths(resolved_root, run_name=run_name)
            cv_path = batch_paths.results_dir / "cv_leaderboard.csv"
            final_path = batch_paths.results_dir / "final_selected_models.csv"
            if not cv_path.exists():
                raise FileNotFoundError(f"Missing required results file: {cv_path}")
            if not final_path.exists():
                raise FileNotFoundError(f"Missing required results file: {final_path}")
            cv_frames.append(pd.read_csv(cv_path))
            final_frames.append(pd.read_csv(final_path))

        cv_frame = _merge_result_frames(cv_frames, ["trial", "model", "preprocessing"] )
        cv_frame = cv_frame.sort_values(
            ["trial", "model", "best_cv_accuracy"],
            ascending=[True, True, False],
        ).reset_index(drop=True)

        final_frame = _merge_result_frames(final_frames, ["trial", "model"] )
        final_frame = final_frame.sort_values(["model", "trial"]).reset_index(drop=True)
        summary_frame, email_frame = _build_summary_frames(final_frame)
        _save_result_tables(
            cv_frame,
            final_frame,
            summary_frame,
            email_frame,
            combined_paths,
            source_runs=selected_run_names,
        )

        return {
            "cv": cv_frame,
            "final": final_frame,
            "summary": summary_frame,
            "email": email_frame,
        }


BATCH_PRESETS = {
    "light": {
        "run_name": "batch_light",
        "selected_model_names": [
            "knn_1",
            "logistic_regression_ova",
            "linear_svm_ova",
        ],
    },
    "heavy": {
        "run_name": "batch_heavy",
        "selected_model_names": [
            "rbf_svm_ova",
            "random_forest",
            "mlp",
        ],
    },
    "rbf_only": {
        "run_name": "batch_rbf_svm",
        "selected_model_names": ["rbf_svm_ova"],
    },
    "random_forest_only": {
        "run_name": "batch_random_forest",
        "selected_model_names": ["random_forest"],
    },
    "mlp_only": {
        "run_name": "batch_mlp",
        "selected_model_names": ["mlp"],
    },
}

if BATCH_PRESET is not None:
    if BATCH_PRESET not in BATCH_PRESETS:
        raise ValueError(f"Unknown BATCH_PRESET: {BATCH_PRESET}")
    RUN_NAME = BATCH_PRESETS[BATCH_PRESET]["run_name"]
    SELECTED_MODEL_NAMES = BATCH_PRESETS[BATCH_PRESET]["selected_model_names"]

_call_configure_project(root_dir=project_root, grid_search_jobs=GRID_SEARCH_JOBS, run_name=RUN_NAME)
print(project_config.get_runtime_config())
print("Selected trials:", SELECTED_TRIAL_NAMES)
print("Selected models:", SELECTED_MODEL_NAMES)
print("Combine run names:", COMBINE_RUN_NAMES)

{'root_dir': PosixPath('/content/CS5487-Project'), 'grid_search_jobs': 2, 'run_name': None}
Selected trials: None
Selected models: None
Combine run names: ['batch_light', 'batch_heavy']


In [5]:
from digits_project.data import load_digits_project_data

bundle = load_digits_project_data()
print('digits4000 shape:', bundle.X.shape)
print('digits4000 labels:', bundle.y.shape)
print('challenge shape:', bundle.challenge_X.shape)
print('challenge labels:', bundle.challenge_y.shape)
print('trials:', [trial.name for trial in bundle.trials])

digits4000 shape: (4000, 784)
digits4000 labels: (4000,)
challenge shape: (150, 784)
challenge labels: (150,)
trials: ['trial_1', 'trial_2']


In [6]:
import pandas as pd
from digits_project.models import MODEL_SPECS

expected_pairs = {
    (trial.name, model_spec.name)
    for trial in bundle.trials
    for model_spec in MODEL_SPECS
}

results_paths = [project_root / "artifacts" / "results" / "final_selected_models.csv"]
runs_dir = project_root / "artifacts" / "runs"
if runs_dir.exists():
    results_paths.extend(sorted(runs_dir.glob("*/results/final_selected_models.csv")))

completed_pairs = set()
for result_path in results_paths:
    if not result_path.exists():
        continue
    frame = pd.read_csv(result_path)
    completed_pairs.update((row.trial, row.model) for row in frame.itertuples(index=False))
    print(f"{result_path.relative_to(project_root)} -> {len(frame)} rows")

print(f"Completed trial/model pairs: {len(completed_pairs)} / {len(expected_pairs)}")
pending_pairs = sorted(expected_pairs - completed_pairs)
if pending_pairs:
    print("Pending pairs:")
    print(pd.DataFrame(pending_pairs, columns=["trial", "model"]).to_string(index=False))
else:
    print("All trial/model pairs already have saved outputs.")

if runs_dir.exists():
    print("Available batch runs:", sorted(path.name for path in runs_dir.iterdir() if path.is_dir()))
else:
    print("Available batch runs: []")

artifacts/results/final_selected_models.csv -> 1 rows
artifacts/runs/batch_heavy/results/final_selected_models.csv -> 6 rows
artifacts/runs/batch_light/results/final_selected_models.csv -> 6 rows
Completed trial/model pairs: 12 / 12
All trial/model pairs already have saved outputs.
Available batch runs: ['batch_heavy', 'batch_light']


In [8]:
results = None
if RUN_EXPERIMENTS:
    results = _call_run_project_experiments(
        root_dir=project_root,
        grid_search_jobs=GRID_SEARCH_JOBS,
        selected_trial_names=SELECTED_TRIAL_NAMES,
        selected_model_names=SELECTED_MODEL_NAMES,
        run_name=RUN_NAME,
    )
else:
    print("RUN_EXPERIMENTS is False, so this notebook will not start a new training run.")

RUN_EXPERIMENTS is False, so this notebook will not start a new training run.


In [9]:
from pathlib import Path

if results is not None:
    print(results["summary"].to_string(index=False))
    print()
    print(results["email"].to_string(index=False))
    print()
    print("Run results saved under:", project_config.PATHS.results_dir)
else:
    print("No new experiment run was executed in this session.")

if COMBINE_RUN_NAMES:
    combined_results = combine_experiment_runs(root_dir=project_root, run_names=COMBINE_RUN_NAMES)
    print()
    print("Combined summary")
    print(combined_results["summary"].to_string(index=False))
    print()
    print("Combined results saved under:", Path(project_root) / "artifacts" / "results")
elif RUN_NAME is not None:
    print()
    print("Set COMBINE_RUN_NAMES to completed batch names and rerun this cell to rebuild artifacts/results.")
elif not RUN_EXPERIMENTS:
    print()
    print("Set COMBINE_RUN_NAMES to completed batch names if you only want to combine prior batch runs.")

No new experiment run was executed in this session.

Combined summary
                  model  mnist_accuracy_mean  mnist_accuracy_std  challenge_accuracy_mean  challenge_accuracy_std  mnist_macro_f1_mean  challenge_macro_f1_mean
            rbf_svm_ova              0.94700            0.001414                 0.750000                0.042426             0.946871                 0.737752
          random_forest              0.92550            0.004950                 0.703333                0.023570             0.925327                 0.685171
                  knn_1              0.91600            0.003536                 0.683333                0.032998             0.915482                 0.675735
                    mlp              0.91450            0.010607                 0.660000                0.047140             0.914243                 0.649431
logistic_regression_ova              0.88450            0.008485                 0.626667                0.000000             0.88

## Challenge Reminder

Keep challenge accuracy out of presentation and poster material. The notebook can compute and save the private challenge results, but those numbers are only for the report and the instructor email workflow.